# Cluster Comparison: UPSW_neo vs UPSW_V2

Compares the two obstacle-clustering approaches on the same input LAZ files.

| | **neo** (`3. Extract 2D obstacles.ipynb`) | **V2** (`4. Auto-Labeling Pipeline.ipynb`) |
|---|---|---|
| **Algorithm** | 3-D connected components | 2-D DBSCAN on voxel footprints |
| **Candidate points** | ALL points above height threshold | Only **unknown** (label = 0) points |
| **Voxel** | 0.25 m (3-D) | 0.25 m (2-D footprint) |
| **Ground grid** | 0.5 m | 0.5 m |
| **Height threshold** | 0.25 m | 0.25 m |
| **Min cluster** | 30 voxels | 5 voxels + 0.2 m² area |
| **DBSCAN ε** | — | 0.6 m |
| **Area filter** | none | 0.2 – 40 m² |
| **Polygon** | convex hull\* | convex hull |

\* neo originally uses concave hull (alphashape), but convex hull is used here for a fair comparison.

**Key design difference:** neo sees everything above ground (including already-identified trees, cars, etc.); V2 sees only the points the BGT pipeline could not identify.

## Config

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import laspy
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from shapely.geometry import MultiPoint
from scipy.ndimage import label as cc_label

V2_ROOT  = Path('/Users/minkeverweij/UPSW_V2')
NEO_ROOT = Path('/Users/minkeverweij/UPSW_neo')

sys.path.insert(0, str(V2_ROOT))
from utils.obstacle_extractor_2d import (
    build_ground_grid, compute_heights,
    VOXEL_2D, DBSCAN_EPS, DBSCAN_MIN_VOXELS,
    HEIGHT_THRESHOLD, GROUND_GRID_SIZE, MIN_AREA, MAX_AREA,
    extract_obstacles as v2_extract,
    clusters_to_polygons as v2_to_polygons,
)

LABELED_DIR = V2_ROOT / 'data/output/labeled_pointcloud'

# Tiles present in both projects
TILECODES = ['120300_489300', '120300_488900']
CRS = 'EPSG:28992'

# Neo algorithm constants (from obstacles_utils.py)
NEO_VOXEL    = 0.25   # m, 3-D
NEO_MIN_COMP = 30     # min voxels per component

# Plot colours
NEO_COLOR = '#4da6ff'   # blue
V2_COLOR  = '#ff7043'   # red-orange
TILE_LABEL_COLORS = {0: '#1e1e1e', 1: '#e8d44d', 9: '#6aaa6a', 10: '#666666'}

## Algorithm implementations

In [ ]:
def _load_laz(laz_path):
    pc = laspy.read(str(laz_path))
    xyz = np.column_stack([
        np.asarray(pc.x, dtype=np.float64),
        np.asarray(pc.y, dtype=np.float64),
        np.asarray(pc.z, dtype=np.float64),
    ])
    lbl = (
        np.asarray(pc.label, dtype=np.int32)
        if 'label' in pc.point_format.extra_dimension_names
        else np.zeros(len(xyz), dtype=np.int32)
    )
    return xyz, lbl


def _pts_to_gdf(clusters_pts, crs, min_area=0.0, max_area=np.inf):
    """Convert list of (N,3) point arrays to a GeoDataFrame of convex hull polygons."""
    polys = []
    for pts in clusters_pts:
        if len(pts) < 3:
            continue
        hull = MultiPoint(pts[:, :2]).convex_hull
        if not hull.is_valid:
            continue
        if hull.geom_type not in ('Polygon', 'MultiPolygon'):
            continue
        area = hull.area
        if min_area <= area <= max_area:
            polys.append({'geometry': hull, 'area_m2': area,
                          'centroid_x': hull.centroid.x,
                          'centroid_y': hull.centroid.y})
    return gpd.GeoDataFrame(polys, crs=crs) if polys else gpd.GeoDataFrame(
        columns=['geometry', 'area_m2', 'centroid_x', 'centroid_y'], crs=crs
    )


def neo_cluster(xyz, lbl, verbose=True):
    """
    Neo approach: 3-D connected components on ALL points above height threshold.
    Uses vectorised ground grid (same as V2) for speed.
    """
    def log(msg):
        if verbose: print(msg)

    grid, xmin, ymin = build_ground_grid(xyz, lbl)
    if grid is None:
        log('  No ground points — skipping')
        return []
    heights = compute_heights(xyz, grid, xmin, ymin)

    # ALL points above threshold (not filtered by label)
    obs_mask = heights > HEIGHT_THRESHOLD
    obs_xyz  = xyz[obs_mask]
    log(f'  Obstacle candidates: {len(obs_xyz):,}')

    # 3-D voxelisation
    v    = np.floor(obs_xyz / NEO_VOXEL).astype(np.int32)
    vmin = v.min(axis=0)
    v   -= vmin
    vgrid = np.zeros(tuple(v.max(axis=0) + 1), dtype=np.uint8)
    vgrid[v[:, 0], v[:, 1], v[:, 2]] = 1

    # 3-D connected components (26-connectivity)
    struct = np.ones((3, 3, 3), dtype=np.int8)
    labeled_vol, n_comp = cc_label(vgrid, struct)
    log(f'  Components found: {n_comp}')

    clusters = []
    for cid in range(1, n_comp + 1):
        vox = np.argwhere(labeled_vol == cid)
        if len(vox) < NEO_MIN_COMP:
            continue
        pts = (vox + vmin).astype(np.float64) * NEO_VOXEL
        clusters.append(pts)
    log(f'  Clusters (≥{NEO_MIN_COMP} voxels): {len(clusters)}')
    return clusters


def v2_cluster(xyz, lbl, verbose=True):
    """
    V2 approach: 2-D DBSCAN on UNKNOWN (label=0) points above height threshold.
    """
    # v2_extract works from a laz file; we call the shared primitives directly
    from sklearn.cluster import DBSCAN

    def log(msg):
        if verbose: print(msg)

    grid, xmin, ymin = build_ground_grid(xyz, lbl)
    if grid is None:
        log('  No ground points — skipping')
        return []
    heights = compute_heights(xyz, grid, xmin, ymin)

    # Only UNKNOWN (label=0) points
    obs_mask = (lbl == 0) & (heights > HEIGHT_THRESHOLD)
    obs_xyz  = xyz[obs_mask]
    log(f'  Obstacle candidates (unknown only): {len(obs_xyz):,}')

    if len(obs_xyz) < DBSCAN_MIN_VOXELS:
        log('  Too few candidates')
        return []

    # 2-D voxelisation
    vx = np.floor(obs_xyz[:, 0] / VOXEL_2D).astype(np.int32)
    vy = np.floor(obs_xyz[:, 1] / VOXEL_2D).astype(np.int32)
    cells_rc, inverse = np.unique(np.column_stack([vx, vy]), axis=0, return_inverse=True)
    cell_centers = cells_rc * VOXEL_2D + VOXEL_2D / 2.0
    log(f'  2-D voxels: {len(cell_centers):,}')

    db = DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_VOXELS,
                algorithm='ball_tree', n_jobs=-1)
    cell_ids = db.fit_predict(cell_centers)
    point_ids = cell_ids[inverse]

    n_clusters = int(cell_ids.max()) + 1
    log(f'  Clusters found: {n_clusters}')

    clusters = [obs_xyz[point_ids == cid] for cid in range(n_clusters)]
    return clusters

## Run both methods

Uses the shared `bgt_labeled_<tilecode>.laz` files from UPSW_V2.
The large tile (120300_489300, ~26 M points) takes ~1–2 minutes for the neo 3-D voxelisation step.

In [ ]:
results = {}

for tilecode in TILECODES:
    laz_path = LABELED_DIR / f'bgt_labeled_{tilecode}.laz'
    if not laz_path.exists():
        print(f'  [skip] {tilecode} — LAZ not found: {laz_path}')
        continue

    print(f'\n══ {tilecode} ══')
    print(f'  Loading {laz_path.name} …')
    xyz, lbl = _load_laz(laz_path)
    print(f'  Points: {len(xyz):,}  |  label counts: '
          f'{dict(zip(*np.unique(lbl, return_counts=True)))}')

    print('\n  [neo] 3-D connected components …')
    neo_pts  = neo_cluster(xyz, lbl)
    neo_gdf  = _pts_to_gdf(neo_pts, CRS)  # no area filter (neo has none)

    print('\n  [V2]  2-D DBSCAN (unknown only) …')
    v2_pts   = v2_cluster(xyz, lbl)
    v2_gdf   = _pts_to_gdf(v2_pts, CRS, min_area=MIN_AREA, max_area=MAX_AREA)

    results[tilecode] = {'neo': neo_gdf, 'v2': v2_gdf, 'xyz': xyz, 'lbl': lbl}
    print(f'\n  neo polygons: {len(neo_gdf)}  |  V2 polygons: {len(v2_gdf)}')

print('\nDone.')

## Tile context (road/ground background)

In [ ]:
def _load_tile_bg(tilecode, max_pts=80_000):
    """Load road_ground_labeled LAZ subsampled for background rendering."""
    p = V2_ROOT / f'data/output/labeled_pointcloud/road_ground_labeled_{tilecode}.laz'
    if not p.exists():
        return None, None, None
    laz  = laspy.read(str(p))
    step = max(1, len(laz.x) // max_pts)
    return (np.array(laz.x[::step], dtype=np.float32),
            np.array(laz.y[::step], dtype=np.float32),
            np.array(laz['label'][::step], dtype=np.uint8))


def _paint_bg(ax, x, y, lbls):
    if x is None:
        return
    for lbl in np.unique(lbls):
        mask  = lbls == lbl
        color = TILE_LABEL_COLORS.get(int(lbl), '#444')
        alpha = 0.15 if lbl == 0 else 0.65
        size  = 0.2  if lbl == 0 else 0.6
        ax.scatter(x[mask], y[mask], c=color, s=size, linewidths=0,
                   alpha=alpha, rasterized=True, zorder=1)


def _style(ax):
    ax.set_facecolor('#1a1a1a')
    for sp in ax.spines.values():
        sp.set_edgecolor('#444')
    ax.tick_params(labelsize=7, colors='grey')


# Pre-load tile backgrounds
bg = {tc: _load_tile_bg(tc) for tc in results}
print('Tile backgrounds loaded.')

## Plot 1 — Side-by-side cluster maps

In [ ]:
n_tiles = len(results)
fig, axes = plt.subplots(n_tiles, 2, figsize=(16, 7 * n_tiles))
fig.patch.set_facecolor('#1a1a1a')
if n_tiles == 1:
    axes = axes[np.newaxis, :]  # always 2-D

for row, tilecode in enumerate(results):
    neo_gdf = results[tilecode]['neo']
    v2_gdf  = results[tilecode]['v2']
    bx, by, bl = bg[tilecode]

    for col, (gdf, label_txt, color) in enumerate([
        (neo_gdf, f'neo — 3D connected components  (n = {len(neo_gdf)})', NEO_COLOR),
        (v2_gdf,  f'V2 — 2D DBSCAN, unknown only   (n = {len(v2_gdf)})',  V2_COLOR),
    ]):
        ax = axes[row, col]
        _style(ax)
        _paint_bg(ax, bx, by, bl)

        for geom in gdf.geometry:
            if geom.geom_type == 'Polygon':
                xs, ys = geom.exterior.xy
                ax.fill(xs, ys, alpha=0.35, fc=color, ec=color,
                        linewidth=1, zorder=3)
        if len(gdf):
            ax.scatter(gdf.centroid.x, gdf.centroid.y, c=color, s=18,
                       zorder=4, edgecolors='white', linewidths=0.4)

        ax.set_aspect('equal')
        ax.set_title(f'{tilecode}  |  {label_txt}', color='white', fontsize=9)
        ax.set_xlabel('X  (RD New)', color='grey', fontsize=7)
        ax.set_ylabel('Y  (RD New)', color='grey', fontsize=7)

        patches = [
            mpatches.Patch(color='#e8d44d', label='Road'),
            mpatches.Patch(color='#6aaa6a', label='Ground'),
            mpatches.Patch(color=color, label='Cluster'),
        ]
        ax.legend(handles=patches, loc='lower right', fontsize=6,
                  facecolor='#2a2a2a', edgecolor='#555', labelcolor='white')

plt.tight_layout(pad=1)
plt.show()

## Plot 2 — Overlay (both methods on same axes)

In [ ]:
n_tiles = len(results)
fig, axes = plt.subplots(1, n_tiles, figsize=(9 * n_tiles, 7))
fig.patch.set_facecolor('#1a1a1a')
if n_tiles == 1:
    axes = [axes]

for ax, tilecode in zip(axes, results):
    _style(ax)
    _paint_bg(ax, *bg[tilecode])

    neo_gdf = results[tilecode]['neo']
    v2_gdf  = results[tilecode]['v2']

    for geom in neo_gdf.geometry:
        if geom.geom_type == 'Polygon':
            xs, ys = geom.exterior.xy
            ax.fill(xs, ys, alpha=0.25, fc=NEO_COLOR, ec=NEO_COLOR,
                    linewidth=1.2, zorder=3)
    for geom in v2_gdf.geometry:
        if geom.geom_type == 'Polygon':
            xs, ys = geom.exterior.xy
            ax.fill(xs, ys, alpha=0.25, fc=V2_COLOR, ec=V2_COLOR,
                    linewidth=1.2, zorder=4)

    ax.set_aspect('equal')
    ax.set_title(f'{tilecode}  —  neo (blue) vs V2 (red)',
                 color='white', fontsize=10)
    ax.set_xlabel('X  (RD New)', color='grey', fontsize=7)
    ax.set_ylabel('Y  (RD New)', color='grey', fontsize=7)

    patches = [
        mpatches.Patch(color='#e8d44d', label='Road'),
        mpatches.Patch(color='#6aaa6a', label='Ground'),
        mpatches.Patch(color=NEO_COLOR, label=f'neo 3D-CC  (n={len(neo_gdf)})'),
        mpatches.Patch(color=V2_COLOR,  label=f'V2 2D-DBSCAN  (n={len(v2_gdf)})'),
    ]
    ax.legend(handles=patches, loc='lower right', fontsize=7,
              facecolor='#2a2a2a', edgecolor='#555', labelcolor='white')

plt.tight_layout(pad=1)
plt.show()

## Plot 3 — Statistics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.patch.set_facecolor('#1a1a1a')
for ax in axes:
    _style(ax)

# 1) Cluster counts per tile
ax = axes[0]
x = np.arange(len(results))
w = 0.35
neo_counts = [len(results[tc]['neo']) for tc in results]
v2_counts  = [len(results[tc]['v2'])  for tc in results]
ax.bar(x - w/2, neo_counts, w, color=NEO_COLOR, label='neo 3D-CC')
ax.bar(x + w/2, v2_counts,  w, color=V2_COLOR,  label='V2 2D-DBSCAN')
for i, (nc, vc) in enumerate(zip(neo_counts, v2_counts)):
    ax.text(i - w/2, nc + 0.3, str(nc), color='white', ha='center', fontsize=8)
    ax.text(i + w/2, vc + 0.3, str(vc), color='white', ha='center', fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(list(results), fontsize=7, rotation=10)
ax.set_ylabel('Cluster count', color='grey', fontsize=8)
ax.set_title('Cluster count per tile', color='white', fontsize=9)
ax.legend(fontsize=8, facecolor='#2a2a2a', edgecolor='#555', labelcolor='white')

# 2) Area distribution histogram
ax = axes[1]
neo_all = pd.concat([results[tc]['neo'] for tc in results if len(results[tc]['neo'])])
v2_all  = pd.concat([results[tc]['v2']  for tc in results if len(results[tc]['v2'])])
bins = np.linspace(0, 40, 40)
if len(neo_all) and 'area_m2' in neo_all.columns:
    ax.hist(neo_all['area_m2'].dropna(), bins=bins, color=NEO_COLOR, alpha=0.7,
            label='neo 3D-CC')
if len(v2_all) and 'area_m2' in v2_all.columns:
    ax.hist(v2_all['area_m2'].dropna(),  bins=bins, color=V2_COLOR,  alpha=0.7,
            label='V2 2D-DBSCAN')
ax.set_xlabel('Cluster area  (m²)', color='grey', fontsize=8)
ax.set_ylabel('Count', color='grey', fontsize=8)
ax.set_title('Footprint area distribution', color='white', fontsize=9)
ax.legend(fontsize=8, facecolor='#2a2a2a', edgecolor='#555', labelcolor='white')

# 3) Centroid scatter — all tiles combined
ax = axes[2]
for tc in results:
    neo_gdf = results[tc]['neo']
    v2_gdf  = results[tc]['v2']
    if len(neo_gdf):
        ax.scatter(neo_gdf.centroid.x, neo_gdf.centroid.y,
                   c=NEO_COLOR, s=14, linewidths=0, alpha=0.85, zorder=3)
    if len(v2_gdf):
        ax.scatter(v2_gdf.centroid.x,  v2_gdf.centroid.y,
                   c=V2_COLOR,  s=14, linewidths=0, alpha=0.85, zorder=4)
ax.set_aspect('equal')
ax.set_title('Centroid positions (all tiles)', color='white', fontsize=9)
ax.set_xlabel('X  (RD New)', color='grey', fontsize=7)
ax.set_ylabel('Y  (RD New)', color='grey', fontsize=7)
patches = [
    mpatches.Patch(color=NEO_COLOR, label='neo 3D-CC'),
    mpatches.Patch(color=V2_COLOR,  label='V2 2D-DBSCAN'),
]
ax.legend(handles=patches, fontsize=8,
          facecolor='#2a2a2a', edgecolor='#555', labelcolor='white')

plt.tight_layout(pad=0.5)
plt.show()

# Summary table
stats = []
for tc in results:
    ng = results[tc]['neo']
    vg = results[tc]['v2']
    stats.append({
        'tilecode':       tc,
        'neo_n':          len(ng),
        'v2_n':           len(vg),
        'neo_area_med':   round(ng['area_m2'].median(), 2) if len(ng) else None,
        'v2_area_med':    round(vg['area_m2'].median(),  2) if len(vg) else None,
        'neo_area_max':   round(ng['area_m2'].max(),     2) if len(ng) else None,
        'v2_area_max':    round(vg['area_m2'].max(),     2) if len(vg) else None,
    })
display(pd.DataFrame(stats).set_index('tilecode'))

## Spatial overlap analysis

For each cluster in one method: does it spatially intersect any cluster from the other method?
This reveals what each method detects that the other misses.

In [ ]:
overlap_rows = []

for tilecode in results:
    neo_gdf = results[tilecode]['neo'].reset_index(drop=True)
    v2_gdf  = results[tilecode]['v2'].reset_index(drop=True)

    if len(neo_gdf) == 0 or len(v2_gdf) == 0:
        print(f'{tilecode}: one method produced no clusters — skipping overlap')
        continue

    neo_join = gpd.sjoin(neo_gdf[['geometry']], v2_gdf[['geometry']],
                         how='left', predicate='intersects')
    v2_join  = gpd.sjoin(v2_gdf[['geometry']],  neo_gdf[['geometry']],
                         how='left', predicate='intersects')

    neo_matched = neo_join['index_right'].notna().groupby(level=0).any()
    v2_matched  = v2_join['index_right'].notna().groupby(level=0).any()

    n_neo_match = int(neo_matched.sum())
    n_v2_match  = int(v2_matched.sum())
    n_neo_only  = int((~neo_matched).sum())
    n_v2_only   = int((~v2_matched).sum())

    print(f'\n{tilecode}:')
    print(f'  neo total:            {len(neo_gdf):>4}')
    print(f'  V2  total:            {len(v2_gdf):>4}')
    print(f'  neo overlaps a V2 cluster: {n_neo_match:>3}  ({100*n_neo_match/max(len(neo_gdf),1):.0f}%)')
    print(f'  V2  overlaps a neo cluster: {n_v2_match:>3}  ({100*n_v2_match/max(len(v2_gdf),1):.0f}%)')
    print(f'  neo-only (no V2 match):     {n_neo_only}')
    print(f'  V2-only  (no neo match):    {n_v2_only}')

    overlap_rows.append({
        'tilecode':    tilecode,
        'neo_total':   len(neo_gdf),
        'v2_total':    len(v2_gdf),
        'neo_in_v2_%': round(100*n_neo_match/max(len(neo_gdf),1)),
        'v2_in_neo_%': round(100*n_v2_match/max(len(v2_gdf),1)),
        'neo_only':    n_neo_only,
        'v2_only':     n_v2_only,
    })

if overlap_rows:
    display(pd.DataFrame(overlap_rows).set_index('tilecode'))

## Plot 4 — neo-only vs V2-only clusters

In [ ]:
n_tiles = len([tc for tc in results
               if len(results[tc]['neo']) and len(results[tc]['v2'])])
if n_tiles == 0:
    print('No tiles with both methods — skipping.')
else:
    fig, axes = plt.subplots(1, n_tiles, figsize=(9 * n_tiles, 7))
    fig.patch.set_facecolor('#1a1a1a')
    if n_tiles == 1:
        axes = [axes]

    valid_tiles = [tc for tc in results
                   if len(results[tc]['neo']) and len(results[tc]['v2'])]

    for ax, tilecode in zip(axes, valid_tiles):
        _style(ax)
        _paint_bg(ax, *bg[tilecode])

        neo_gdf = results[tilecode]['neo'].reset_index(drop=True)
        v2_gdf  = results[tilecode]['v2'].reset_index(drop=True)

        neo_join = gpd.sjoin(neo_gdf[['geometry']], v2_gdf[['geometry']],
                             how='left', predicate='intersects')
        v2_join  = gpd.sjoin(v2_gdf[['geometry']],  neo_gdf[['geometry']],
                             how='left', predicate='intersects')
        neo_has = neo_join['index_right'].notna().groupby(level=0).any()
        v2_has  = v2_join['index_right'].notna().groupby(level=0).any()

        neo_only = neo_gdf[~neo_has.reindex(neo_gdf.index, fill_value=False)]
        v2_only  = v2_gdf[~v2_has.reindex(v2_gdf.index, fill_value=False)]
        shared_neo = neo_gdf[neo_has.reindex(neo_gdf.index, fill_value=False)]

        for geom in shared_neo.geometry:
            if geom.geom_type == 'Polygon':
                xs, ys = geom.exterior.xy
                ax.fill(xs, ys, alpha=0.3, fc='#aaaaaa', ec='#aaaaaa',
                        linewidth=1, zorder=3)
        for geom in neo_only.geometry:
            if geom.geom_type == 'Polygon':
                xs, ys = geom.exterior.xy
                ax.fill(xs, ys, alpha=0.4, fc=NEO_COLOR, ec=NEO_COLOR,
                        linewidth=1.2, zorder=4)
        for geom in v2_only.geometry:
            if geom.geom_type == 'Polygon':
                xs, ys = geom.exterior.xy
                ax.fill(xs, ys, alpha=0.4, fc=V2_COLOR, ec=V2_COLOR,
                        linewidth=1.2, zorder=5)

        ax.set_aspect('equal')
        ax.set_title(f'{tilecode}  —  unique vs shared clusters', color='white', fontsize=10)
        ax.set_xlabel('X  (RD New)', color='grey', fontsize=7)
        ax.set_ylabel('Y  (RD New)', color='grey', fontsize=7)

        patches = [
            mpatches.Patch(color='#e8d44d', label='Road'),
            mpatches.Patch(color='#6aaa6a', label='Ground'),
            mpatches.Patch(color='#aaaaaa', label=f'Both  ({len(shared_neo)})'),
            mpatches.Patch(color=NEO_COLOR, label=f'neo only  ({len(neo_only)})'),
            mpatches.Patch(color=V2_COLOR,  label=f'V2 only  ({len(v2_only)})'),
        ]
        ax.legend(handles=patches, loc='lower right', fontsize=7,
                  facecolor='#2a2a2a', edgecolor='#555', labelcolor='white')

    plt.tight_layout(pad=1)
    plt.show()